
# Contributor: Deepak Avachitkar
# Project: Military Analytics Dashboard (Web scraping)

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

In [3]:
# User-Agent header to avoid basic blocking
HEADERS = {"User-Agent": "Mozilla/5.0"}


### Function: read_links_txt
### Reads metric URLs from a TXT file
### Cleans bullets/arrows and returns valid URLs

In [5]:
def read_links_txt(path):
    links = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            # Remove bullets and arrow symbols if present
            line = line.replace("- ", "")
            if "→" in line:
                line = line.split("→")[0].strip()

            # Keep only valid URLs
            if line.startswith("http"):
                links.append(line)

    return links


### Function: scrape_global_firepower_data
### 1. Scrapes base country ranking data
### 2. Scrapes multiple metric pages from TXT file
### 3. Merges all metrics country-wise
### 4. Cleans numeric values

In [9]:
def scrape_global_firepower_data():

    base_url = "https://www.globalfirepower.com/countries-listing.php"

    links_file = r"C:\Users\avach\links_for_military_data.txt"
    metric_urls = read_links_txt(links_file)

    # ---- Base ranking scrape ----
    r = requests.get(base_url, headers=HEADERS, timeout=30)
    soup = BeautifulSoup(r.text, "html.parser")

    containers = soup.find_all(
        "div", class_="picTrans recordsetContainer boxShadow zoom"
    )

    ranks, countries = [], []

    for item in containers:
        try:
            ranks.append(
                item.find(
                    "span", class_="textWhite textLarge textBold"
                ).text.strip()
            )
            countries.append(
                item.find(
                    "span", class_="textWhite textLarge textShadow"
                ).text.strip()
            )
        except:
            continue

    df_main = pd.DataFrame({
        "rank": ranks,
        "country": countries
    })

    # ---- Metric pages scrape ----
    for url in metric_urls:

        column_name = url.split("/")[-1].replace(".php", "")

        r = requests.get(url, headers=HEADERS, timeout=30)
        soup = BeautifulSoup(r.text, "html.parser")

        containers = soup.find_all(
            "div", class_="picTrans recordsetContainer boxShadow zoom"
        )

        sub_countries, values = [], []

        for item in containers:
            try:
                sub_countries.append(
                    item.find(
                        "span", class_="textWhite textLarge textShadow"
                    ).text.strip()
                )
                values.append(
                    item.find_all(
                        "span", class_="textWhite textLarge"
                    )[-1].text.strip()
                )
            except:
                continue

        df_sub = pd.DataFrame({
            "country": sub_countries,
            column_name: values
        })

        df_main = df_main.merge(df_sub, on="country", how="left")

    # ---- Clean numeric columns ----
    for col in df_main.columns[2:]:
        cleaned = []

        for val in df_main[col]:
            if pd.isna(val):
                cleaned.append(None)
                continue

            v = str(val).replace(",", "").replace(" ", "")
            m = re.search(r"-?\d+\.?\d*", v)
            cleaned.append(float(m.group()) if m else None)

        df_main[col] = cleaned

    return df_main


In [14]:
df = scrape_global_firepower_data()
df.head()


,rank,country,total-population-by-country,available-military-manpower,manpower-fit-for-military-service,manpower-reaching-military-age-annually,active-military-manpower,active-reserve-military-manpower,manpower-paramilitary,aircraft-total,...,natural-gas-production-by-country,natural-gas-consumption-by-country,proven-natural-gas-reserves-by-country,coal-production-by-country,coal-consumption-by-country,proven-coal-reserves-by-country,square-land-area,coastline-coverage,border-coverage,waterway-coverage
0,1,United States,3.419634e+08,150463900.0,124816644.0,4445524.0,1328000.0,799500.0,0.0,13043.0,...,1.029000e+12,9.143010e+11,1.340200e+13,5.488490e+08,4.760440e+08,2.489410e+11,9833517.0,19924.0,12002.0,41009.0
1,2,Russia,1.408208e+08,69002197.0,46189226.0,1267387.0,1320000.0,2000000.0,250000.0,4292.0,...,6.178300e+11,4.722390e+11,4.780500e+13,5.081900e+08,3.109580e+08,1.621660e+11,17098242.0,37653.0,22407.0,102000.0
2,3,China,1.415043e+09,764123366.0,626864169.0,19810606.0,2035000.0,510000.0,625000.0,3309.0,...,2.253410e+11,3.661600e+11,6.654000e+12,4.827000e+09,5.313000e+09,1.431970e+11,9596960.0,14500.0,22457.0,27700.0
3,4,India,1.409128e+09,662290299.0,522786598.0,23955181.0,1455550.0,1155000.0,2527000.0,2229.0,...,3.317000e+10,5.886700e+10,1.381000e+12,9.856710e+08,1.200000e+09,1.110520e+11,3287263.0,7000.0,13888.0,14500.0
4,5,South Korea,5.208180e+07,26040900.0,21353538.0,416654.0,600000.0,3100000.0,120000.0,1592.0,...,5.512700e+07,5.948000e+10,7.079000e+09,1.559500e+07,1.364130e+08,3.260000e+08,99720.0,2413.0,237.0,1600.0


In [16]:
df.to_csv("global_firepower_data_2025.csv", index=False)
